# 🎙️ AI Meeting Assistant — Colab launcher

Runs the full app (faster-whisper → Gemini refinement → Gemini documentation) on a Colab GPU and gives you a public link to the Streamlit interface.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Add your Gemini key as a Colab secret: click the 🔑 icon in the left sidebar → *Add new secret* → name `GEMINI_API_KEY`, paste the key, and enable *Notebook access*.
3. Run the cells in order.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - the app will run on CPU with a smaller Whisper model"

## 2. Get the code
Either set `REPO_URL` to your GitHub repository, or leave it empty and upload `meeting-assistant.zip` when prompted.

In [ ]:
import os, zipfile, glob

REPO_URL = ""  # e.g. "https://github.com/<you>/meeting-assistant.git"

os.chdir("/content")
if REPO_URL:
    !rm -rf meeting-assistant && git clone -q {REPO_URL} meeting-assistant
elif not os.path.exists("/content/meeting-assistant/app.py"):
    from google.colab import files
    uploaded = files.upload()  # choose meeting-assistant.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall("/content")
import os, sys, glob
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)  # Colab does not import from the current folder automatically
print("Project folder:", PROJECT)

In [ ]:
# 3. Install dependencies (ffmpeg is already on Colab)
!pip install -q -r requirements.txt

In [ ]:
# 4. Load the Gemini key from Colab secrets
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# Optional overrides:
# os.environ["REFINE_MODEL"] = "gemini-3.5-flash-lite"
# os.environ["DOCS_MODEL"] = "gemini-3.5-flash-lite"
# os.environ["WHISPER_MODEL"] = "large-v3"   # 'auto' = large-v3 on GPU
print("Key loaded:", bool(os.environ["GEMINI_API_KEY"]))

In [ ]:
# 5. (Recommended) Pre-download the Whisper model so the first run in the app is fast (~3 GB for large-v3)
import os, sys, glob
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)  # Colab does not import from the current folder automatically
from meeting_assistant.transcribe import WhisperTranscriber
stt = WhisperTranscriber(os.environ.get("WHISPER_MODEL", "auto"))
print("Loaded:", stt.name)
del stt

In [ ]:
# 6. Run the tests (offline, uses fake models) - should all pass
import os, sys, glob
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)  # Colab does not import from the current folder automatically
!python -m pytest -q

In [ ]:
# 7. Start the app and get links to open it
import os, sys, glob, subprocess, time, re, socket, urllib.request
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)

def app_is_up():
    try:
        return urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=3).read() == b"ok"
    except Exception:
        return False

# --- start Streamlit (only if it is not already running) ---
if not app_is_up():
    subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
         "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
        stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(90):
        if app_is_up():
            break
        time.sleep(1)

if not app_is_up():
    print("❌ The app did not start. Last lines of streamlit.log (send these to get help):\n")
    print(open("streamlit.log").read()[-3000:])
else:
    print("✅ The app is running inside Colab.\n")

    # --- Link A: Colab's own proxy (works only in this browser, while you are signed in) ---
    try:
        from google.colab.output import eval_js
        print("Link A (Colab, open in this same browser):")
        print("  ", eval_js("google.colab.kernel.proxyPort(8501)"), "\n")
    except Exception as e:
        print("Link A unavailable:", e, "\n")

    # --- Link B: public Cloudflare tunnel (works on any device) ---
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])  # drop any old tunnel
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
    url = None
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if m:
            url = m.group(0)
            break
    if url:
        host = url.split("//")[1]
        for _ in range(30):  # a new tunnel takes a little while to become reachable
            try:
                socket.gethostbyname(host)
                break
            except OSError:
                time.sleep(2)
        time.sleep(5)
        print("Link B (public, any device):")
        print("  ", url)
        print("\nIf Link B says 'site can't be reached', wait 30 seconds and refresh. Some providers (e.g. Jio)")
        print("block trycloudflare.com - then use Link A, or switch Wi-Fi/hotspot, or set DNS to 8.8.8.8.")
    else:
        print("Link B could not be created. Use Link A. Tunnel log:\n", open("tunnel.log").read()[-1500:])

## Optional: command-line run
Processes one file end to end and saves every output under `runs/` (useful for the sample-output deliverable).

In [ ]:
import os, sys, glob
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)  # Colab does not import from the current folder automatically
from google.colab import files
up = files.upload()
audio = next(iter(up))
!python cli.py "{audio}" --context "add domain terms and names here"
!ls -la runs/*